# Sector ETF ROI Analysis and Portfolio Plan

This notebook uses the ETF proposal sectors from `Analysis/docs/Full Research.md` and expands each sector to at least 5 ETF choices.

## What this notebook does
- Builds sector ETF universe (5+ ETFs per sector)
- Pulls historical prices
- Computes ROI and risk metrics (1Y/3Y/5Y/10Y total return, CAGR, volatility, Sharpe, max drawdown)
- Ranks ETFs per sector
- Produces a proposed diversified sector plan with suggested weights

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

plt.style.use('seaborn-v0_8')
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 40)

## 1) Sector ETF Universe (Full Research aligned, expanded to 5+ each)

In [ ]:
SECTOR_ETFS = {
    'Consumer Staples': ['XLP', 'VDC', 'IYK', 'PSL', 'FSTA'],
    'Healthcare': ['XLV', 'VHT', 'IYH', 'IHF', 'FHLC'],
    'Industrials': ['XLI', 'VIS', 'IYJ', 'FIDU', 'RGI'],
    'Financials': ['XLF', 'VFH', 'IYF', 'KBE', 'KRE'],
    'Utilities': ['XLU', 'FUTY', 'VPU', 'IDU', 'RYU'],
    'Energy': ['XLE', 'VDE', 'FENY', 'IXC', 'RYE'],
    'Real Estate': ['XLRE', 'VNQ', 'IYR', 'SCHH', 'RWR'],
    'Materials': ['XLB', 'VAW', 'IYM', 'RTM', 'FMAT'],
    'Consumer Discretionary': ['XLY', 'VCR', 'IYC', 'RCD', 'FDIS'],
    'International Equity': ['VXUS', 'VEA', 'VWO', 'IXUS', 'IEFA'],
    'Bonds': ['BND', 'AGG', 'SCHZ', 'IUSB', 'GOVT'],
    'Commodities': ['GLD', 'SLV', 'PDBC', 'DBC', 'IAU'],
}

universe_rows = []
for sector, tickers in SECTOR_ETFS.items():
    for ticker in tickers:
        universe_rows.append({'sector': sector, 'ticker': ticker})

universe_df = pd.DataFrame(universe_rows)
print(f'Total sectors: {universe_df.sector.nunique()}')
print(f'Total ETFs: {len(universe_df)}')
universe_df.head(15)

## 2) Download Price Data

In [ ]:
ALL_TICKERS = sorted(universe_df['ticker'].unique())
START_DATE = '2013-01-01'
END_DATE = datetime.today().strftime('%Y-%m-%d')

price_data = yf.download(
    tickers=ALL_TICKERS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)[['Close']]

if isinstance(price_data.columns, pd.MultiIndex):
    close = price_data['Close'].copy()
else:
    close = price_data.copy()

close = close.dropna(axis=1, how='all').sort_index()
returns = close.pct_change().dropna(how='all')

print(f'Date range: {close.index.min().date()} -> {close.index.max().date()}')
print(f'Usable tickers: {close.shape[1]} / {len(ALL_TICKERS)}')

## 3) ROI and Risk Metric Functions

In [ ]:
RISK_FREE_RATE = 0.04
TRADING_DAYS = 252

def trailing_total_return(series, years):
    if series.dropna().empty:
        return np.nan
    end = series.dropna().index.max()
    start = end - pd.DateOffset(years=years)
    sliced = series[series.index >= start].dropna()
    if len(sliced) < 2:
        return np.nan
    return sliced.iloc[-1] / sliced.iloc[0] - 1

def trailing_cagr(series, years):
    total_ret = trailing_total_return(series, years)
    if pd.isna(total_ret):
        return np.nan
    return (1 + total_ret) ** (1 / years) - 1

def trailing_volatility(ret_series, years):
    if ret_series.dropna().empty:
        return np.nan
    end = ret_series.dropna().index.max()
    start = end - pd.DateOffset(years=years)
    sliced = ret_series[ret_series.index >= start].dropna()
    if len(sliced) < 20:
        return np.nan
    return sliced.std() * np.sqrt(TRADING_DAYS)

def trailing_sharpe(ret_series, years, rf=RISK_FREE_RATE):
    if ret_series.dropna().empty:
        return np.nan
    end = ret_series.dropna().index.max()
    start = end - pd.DateOffset(years=years)
    sliced = ret_series[ret_series.index >= start].dropna()
    if len(sliced) < 20:
        return np.nan
    annual_ret = sliced.mean() * TRADING_DAYS
    annual_vol = sliced.std() * np.sqrt(TRADING_DAYS)
    if annual_vol == 0:
        return np.nan
    return (annual_ret - rf) / annual_vol

def trailing_max_drawdown(series, years):
    if series.dropna().empty:
        return np.nan
    end = series.dropna().index.max()
    start = end - pd.DateOffset(years=years)
    sliced = series[series.index >= start].dropna()
    if len(sliced) < 2:
        return np.nan
    wealth = sliced / sliced.iloc[0]
    drawdown = wealth / wealth.cummax() - 1
    return drawdown.min()

## 4) Build ETF ROI Scorecard

In [ ]:
records = []
for _, row in universe_df.iterrows():
    sector = row['sector']
    ticker = row['ticker']
    if ticker not in close.columns:
        continue

    p = close[ticker].dropna()
    r = returns[ticker].dropna() if ticker in returns.columns else pd.Series(dtype=float)

    records.append({
        'sector': sector,
        'ticker': ticker,
        'total_return_1y': trailing_total_return(p, 1),
        'total_return_3y': trailing_total_return(p, 3),
        'total_return_5y': trailing_total_return(p, 5),
        'total_return_10y': trailing_total_return(p, 10),
        'cagr_3y': trailing_cagr(p, 3),
        'cagr_5y': trailing_cagr(p, 5),
        'cagr_10y': trailing_cagr(p, 10),
        'vol_3y': trailing_volatility(r, 3),
        'sharpe_3y': trailing_sharpe(r, 3),
        'max_drawdown_5y': trailing_max_drawdown(p, 5),
    })

scorecard = pd.DataFrame(records)
scorecard.head()

In [ ]:
# Composite ROI score (higher is better)
# Emphasis on 5Y CAGR + Sharpe, penalty for volatility and drawdown
scorecard['score'] = (
    0.40 * scorecard['cagr_5y'].fillna(0)
    + 0.20 * scorecard['cagr_10y'].fillna(0)
    + 0.25 * scorecard['sharpe_3y'].fillna(0)
    - 0.10 * scorecard['vol_3y'].fillna(0)
    + 0.05 * (1 + scorecard['max_drawdown_5y'].fillna(-1))
)

ranked = scorecard.sort_values(['sector', 'score'], ascending=[True, False]).reset_index(drop=True)
ranked.head(20)

## 5) Top 5 ETF Options Per Sector (ROI-ranked)

In [ ]:
top5_per_sector = ranked.groupby('sector', as_index=False).head(5)

display_cols = [
    'sector', 'ticker', 'score', 'total_return_1y', 'total_return_3y', 'total_return_5y',
    'cagr_5y', 'cagr_10y', 'vol_3y', 'sharpe_3y', 'max_drawdown_5y'
]

top5_per_sector[display_cols].style.format({
    'score': '{:.4f}',
    'total_return_1y': '{:.1%}',
    'total_return_3y': '{:.1%}',
    'total_return_5y': '{:.1%}',
    'cagr_5y': '{:.1%}',
    'cagr_10y': '{:.1%}',
    'vol_3y': '{:.1%}',
    'sharpe_3y': '{:.2f}',
    'max_drawdown_5y': '{:.1%}'
})

## 6) Sector Plan and Suggested ETF Picks

Below we choose a primary ETF per sector (highest score), then apply a strategic sector weight model.

In [ ]:
primary_pick = ranked.groupby('sector', as_index=False).first()[['sector', 'ticker', 'score', 'cagr_5y', 'sharpe_3y', 'max_drawdown_5y']]

sector_weights = {
    'Consumer Staples': 8,
    'Healthcare': 10,
    'Industrials': 9,
    'Financials': 9,
    'Utilities': 6,
    'Energy': 6,
    'Real Estate': 6,
    'Materials': 5,
    'Consumer Discretionary': 10,
    'International Equity': 12,
    'Bonds': 12,
    'Commodities': 7,
}

plan = primary_pick.copy()
plan['target_weight_pct'] = plan['sector'].map(sector_weights)
plan = plan.sort_values('target_weight_pct', ascending=False)

print('Total plan weight:', plan['target_weight_pct'].sum(), '%')
plan

In [ ]:
def reason_text(row):
    cagr = row['cagr_5y']
    sharpe = row['sharpe_3y']
    mdd = row['max_drawdown_5y']

    cagr_txt = 'strong' if pd.notna(cagr) and cagr >= 0.10 else ('moderate' if pd.notna(cagr) and cagr >= 0.05 else 'defensive/low')
    sharpe_txt = 'efficient' if pd.notna(sharpe) and sharpe >= 0.60 else ('balanced' if pd.notna(sharpe) and sharpe >= 0.30 else 'higher-risk')
    dd_txt = 'controlled drawdown' if pd.notna(mdd) and mdd > -0.35 else 'deep drawdown risk'

    return f"5Y CAGR is {cagr_txt}, 3Y risk-adjusted return is {sharpe_txt}, and history shows {dd_txt}."

plan['roi_reasoning'] = plan.apply(reason_text, axis=1)
plan[['sector', 'ticker', 'target_weight_pct', 'cagr_5y', 'sharpe_3y', 'max_drawdown_5y', 'roi_reasoning']]

## 7) Visualizations

In [ ]:
# A) Sector target weights
plt.figure(figsize=(12, 6))
plot_plan = plan.sort_values('target_weight_pct', ascending=True)
plt.barh(plot_plan['sector'], plot_plan['target_weight_pct'])
plt.title('Proposed Sector Weights (%)')
plt.xlabel('Weight (%)')
plt.tight_layout()
plt.show()

In [ ]:
# B) Heatmap of top-1 ETF metrics by sector
heat = plan.set_index('sector')[['cagr_5y', 'sharpe_3y', 'max_drawdown_5y']].copy()

plt.figure(figsize=(10, 7))
sns.heatmap(heat, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Primary ETF Metrics by Sector')
plt.tight_layout()
plt.show()

## 8) Exports
- Top-5 ETF options per sector
- Final primary-pick sector plan with target weights and ROI reasoning

In [ ]:
top5_path = 'Analysis/docs/sector_etf_top5_roi_options.csv'
plan_path = 'Analysis/docs/sector_etf_primary_plan.csv'

top5_per_sector[display_cols].to_csv(top5_path, index=False)
plan.to_csv(plan_path, index=False)

print('Saved:', top5_path)
print('Saved:', plan_path)